# 14r Probe DINOv2 ViT-B/14 VeRi-776 Train

Spec: `docs/subagent-specs/14r-probe.md`.

Train `vit_base_patch14_dinov2.lvd142m` with the proven 09v v17 TransReID-style recipe on VeRi-776: SIE, BNNeck, 4-group JPM, AdamW with LLRD, cosine warmup, AMP fp16, and the four-row final eval suite.

Verdict bands:

| Band | Concat-patch-flip AQE+rerank mAP | R1 | Action |
|---|---:|---:|---|
| WIN | >=91.54% | >=98.33% | Promote as a SOTA-band VeRi candidate |
| MARGINAL | >=90.5% | >=98.0% | Keep as future fusion-stream candidate |
| FAIL | below either marginal threshold | - | Close DINOv2 standalone for VeRi |

In [ ]:
import json
import os
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


pip_install("timm>=1.0,<2.0", "torch>=2.4", "torchvision>=0.19", "scikit-learn", "matplotlib")

import timm
import torch
import torchvision

print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}, indent=2))

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working")
RUN_DIR = OUTPUT_DIR / "14r_probe_dinov2_veri"
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_MAP_PATH = OUTPUT_DIR / "dinov2_vit_b14_veri776_best_mAP.pth"
BEST_R1_PATH = OUTPUT_DIR / "dinov2_vit_b14_veri776_best_R1.pth"
LAST_PATH = OUTPUT_DIR / "dinov2_vit_b14_veri776_last.pth"
TRAIN_LOG_PATH = OUTPUT_DIR / "train_log.json"
EVAL_RESULTS_PATH = OUTPUT_DIR / "eval_results.json"
RECIPE_PATH = OUTPUT_DIR / "recipe.json"
SUMMARY_PATH = OUTPUT_DIR / "14r_probe_summary.json"

MODEL_NAME = "vit_base_patch14_dinov2.lvd142m"
IMG_SIZE = 224
PATCH_SIZE = 14
NUM_PATCHES = 256
FEATURE_DIM = 768
CONCAT_FEATURE_DIM = 1536
NUM_VERI_CAMERAS = 20
EXPECTED_NUM_CLASSES = 576

EPOCHS = 100
WARMUP_EPOCHS = 10
BACKBONE_LR = 3.5e-4
HEAD_LR = 3.5e-3
MIN_LR = 1e-6
WARMUP_START_LR = 1e-7
WEIGHT_DECAY = 1e-4
TRIPLET_MARGIN = 0.3
LABEL_SMOOTHING = 0.1
LAMBDA_GLOBAL_CE = 1.0
LAMBDA_GLOBAL_TRI = 1.0
LAMBDA_JPM_CE = 1.0

P_IDS = 8
K_INSTANCES = 4
BATCH_SIZE = P_IDS * K_INSTANCES
ACCUM_STEPS = 1
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 4
PERIODIC_EVAL_EPOCHS = set(range(20, 101, 10))
MAX_TOTAL_TRAIN_HOURS = 8.0

DINOV2_MEAN = [0.485, 0.456, 0.406]
DINOV2_STD = [0.229, 0.224, 0.225]
ERASE_VALUE = DINOV2_MEAN


def find_veri_root():
    candidates = [
        Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
        Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    ]
    for candidate in candidates:
        roots = [candidate] + list(candidate.glob("**/*")) if candidate.exists() else []
        for root in roots:
            if (root / "image_train").is_dir() and (root / "image_query").is_dir() and (root / "image_test").is_dir():
                return root
    for root, dirs, files in os.walk("/kaggle/input"):
        path = Path(root)
        if (path / "image_train").is_dir() and (path / "image_query").is_dir() and (path / "image_test").is_dir():
            return path
    raise RuntimeError("VeRi-776 not found. Attach abhyudaya12/veri-vehicle-re-identification-dataset.")


VERI_ROOT = find_veri_root()
print("VeRi root:", VERI_ROOT)
print("Device:", DEVICE)

In [ ]:
CAMERA_PATTERN = re.compile(r"^(?P<pid>-?\d+)_c(?P<camid>\d+)")


def parse_split(split_dir, relabel_train=False):
    items = []
    raw_pids = []
    for image_path in sorted(Path(split_dir).glob("*.jpg")):
        match = CAMERA_PATTERN.match(image_path.name)
        if match is None:
            continue
        pid = int(match.group("pid"))
        if pid == -1:
            continue
        camid = int(match.group("camid"))
        if not 1 <= camid <= NUM_VERI_CAMERAS:
            raise RuntimeError(f"Camera ID out of range in {image_path.name}: {camid}")
        items.append({"path": str(image_path), "pid": pid, "camid": camid - 1})
        raw_pids.append(pid)
    if relabel_train:
        pid_to_label = {pid: index for index, pid in enumerate(sorted(set(raw_pids)))}
        for item in items:
            item["pid"] = pid_to_label[item["pid"]]
    return items


train_items = parse_split(VERI_ROOT / "image_train", relabel_train=True)
query_items = parse_split(VERI_ROOT / "image_query", relabel_train=False)
gallery_items = parse_split(VERI_ROOT / "image_test", relabel_train=False)
NUM_CLASSES = len({item["pid"] for item in train_items})
assert NUM_CLASSES == EXPECTED_NUM_CLASSES, f"Expected 576 train IDs, got {NUM_CLASSES}"

resolved_cfg = timm.data.resolve_model_data_config(timm.create_model(MODEL_NAME, pretrained=False, num_classes=0, img_size=IMG_SIZE))
assert tuple(round(value, 8) for value in resolved_cfg["mean"]) == tuple(round(value, 8) for value in DINOV2_MEAN)
assert tuple(round(value, 8) for value in resolved_cfg["std"]) == tuple(round(value, 8) for value in DINOV2_STD)


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.Pad(10),
    T.RandomCrop((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.4), ratio=(0.3, 3.3333333333), value=ERASE_VALUE),
])

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])


class VeRiDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        image = Image.open(item["path"]).convert("RGB")
        return self.transform(image), int(item["pid"]), int(item["camid"]), item["path"]


class PKBatchSampler(Sampler):
    def __init__(self, items, p, k, seed=SEED):
        self.p = int(p)
        self.k = int(k)
        self.seed = int(seed)
        self.epoch = 0
        self.pid_to_indices = defaultdict(list)
        for index, item in enumerate(items):
            self.pid_to_indices[int(item["pid"])].append(index)
        self.pids = sorted(self.pid_to_indices)
        self.batch_size = self.p * self.k

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        pids = np.array(self.pids)
        rng.shuffle(pids)
        batch = []
        for pid in pids.tolist():
            indices = self.pid_to_indices[int(pid)]
            chosen = rng.choice(indices, size=self.k, replace=len(indices) < self.k).tolist()
            batch.extend(chosen)
            if len(batch) == self.batch_size:
                yield batch
                batch = []

    def __len__(self):
        return len(self.pids) // self.p


train_dataset = VeRiDataset(train_items, train_transform)
query_dataset = VeRiDataset(query_items, test_transform)
gallery_dataset = VeRiDataset(gallery_items, test_transform)


def build_train_loader(p=P_IDS, k=K_INSTANCES):
    sampler = PKBatchSampler(train_items, p=p, k=k)
    loader = DataLoader(
        train_dataset,
        batch_sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    return loader, sampler


train_loader, train_sampler = build_train_loader(P_IDS, K_INSTANCES)
query_loader = DataLoader(query_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
gallery_loader = DataLoader(gallery_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(json.dumps({
    "train_images": len(train_items),
    "query_images": len(query_items),
    "gallery_images": len(gallery_items),
    "num_classes": NUM_CLASSES,
    "num_cameras": NUM_VERI_CAMERAS,
    "p": P_IDS,
    "k": K_INSTANCES,
    "batch_size": BATCH_SIZE,
    "dinov2_mean": DINOV2_MEAN,
    "dinov2_std": DINOV2_STD,
}, indent=2))

In [ ]:
class TransReIDDINOv2ViTB14(nn.Module):
    def __init__(self, num_classes, num_cameras=20, pretrained=True):
        super().__init__()
        self.model_name = MODEL_NAME
        self.num_classes = int(num_classes)
        self.num_cameras = int(num_cameras)
        self.vit = timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=0, img_size=IMG_SIZE)
        self.vit_dim = int(self.vit.embed_dim)
        self.num_blocks = len(self.vit.blocks)
        self.num_patches = int(self.vit.patch_embed.num_patches)
        self.jpm_groups = 4
        self.patches_per_group = self.num_patches // self.jpm_groups
        assert self.vit_dim == FEATURE_DIM, f"embed_dim={self.vit_dim}, expected {FEATURE_DIM}"
        assert self.num_patches == NUM_PATCHES, f"num_patches={self.num_patches}, expected {NUM_PATCHES}"
        assert self.num_patches % self.jpm_groups == 0

        self.sie_embed = nn.Parameter(torch.zeros(self.num_cameras, 1, self.vit_dim))
        nn.init.trunc_normal_(self.sie_embed, std=0.02)

        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)
        self.classifier = nn.Linear(self.vit_dim, self.num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.jpm_bns = nn.ModuleList([nn.BatchNorm1d(self.vit_dim) for _ in range(self.jpm_groups)])
        self.jpm_classifiers = nn.ModuleList([nn.Linear(self.vit_dim, self.num_classes, bias=False) for _ in range(self.jpm_groups)])
        for bn in self.jpm_bns:
            bn.bias.requires_grad_(False)
        for classifier in self.jpm_classifiers:
            nn.init.normal_(classifier.weight, std=0.001)

        print(json.dumps({
            "model_name": self.model_name,
            "embed_dim": self.vit_dim,
            "num_patches": self.num_patches,
            "patches_per_jpm_group": self.patches_per_group,
            "blocks": self.num_blocks,
            "num_classes": self.num_classes,
            "num_cameras": self.num_cameras,
        }, indent=2))

    def forward_tokens(self, images, cam_ids=None):
        batch_size = images.shape[0]
        tokens = self.vit.patch_embed(images)
        if hasattr(self.vit, "_pos_embed"):
            result = self.vit._pos_embed(tokens)
            tokens = result[0] if isinstance(result, tuple) else result
        else:
            cls_token = self.vit.cls_token.expand(batch_size, -1, -1)
            tokens = torch.cat([cls_token, tokens], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, "pos_drop"):
                tokens = self.vit.pos_drop(tokens)
        if cam_ids is not None:
            tokens = tokens + self.sie_embed[cam_ids.long()]
        if hasattr(self.vit, "patch_drop"):
            tokens = self.vit.patch_drop(tokens)
        if hasattr(self.vit, "norm_pre"):
            tokens = self.vit.norm_pre(tokens)
        for block in self.vit.blocks:
            tokens = block(tokens)
        return self.vit.norm(tokens)

    def forward(self, images, cam_ids=None):
        tokens = self.forward_tokens(images, cam_ids=cam_ids)
        global_feat = tokens[:, 0]
        bn_feat = self.bn(global_feat)
        global_logits = self.classifier(bn_feat)
        if self.training:
            patches = tokens[:, 1:]
            shuffle_index = torch.randperm(patches.size(1), device=patches.device)
            shuffled = patches[:, shuffle_index]
            jpm_logits = []
            jpm_feats = []
            for group_index, patch_group in enumerate(torch.chunk(shuffled, self.jpm_groups, dim=1)):
                group_feat = patch_group.mean(dim=1)
                group_bn = self.jpm_bns[group_index](group_feat)
                jpm_feats.append(group_feat)
                jpm_logits.append(self.jpm_classifiers[group_index](group_bn))
            return {
                "global_logits": global_logits,
                "global_feat": global_feat,
                "bn_feat": bn_feat,
                "jpm_logits": jpm_logits,
                "jpm_feats": jpm_feats,
            }
        return F.normalize(bn_feat.float(), p=2, dim=1)

    @torch.no_grad()
    def inference_features(self, images, cam_ids=None, concat_patch=False):
        was_training = self.training
        self.eval()
        tokens = self.forward_tokens(images, cam_ids=cam_ids)
        global_feat = F.normalize(self.bn(tokens[:, 0]).float(), p=2, dim=1)
        if concat_patch:
            patch_tokens = tokens[:, 1:]
            gem_p = 3.0
            patch_feat = (patch_tokens.clamp(min=1e-6) ** gem_p).mean(dim=1) ** (1.0 / gem_p)
            patch_feat = F.normalize(patch_feat.float(), p=2, dim=1)
            global_feat = torch.cat([global_feat, patch_feat], dim=1)
        if was_training:
            self.train()
        return global_feat


model = TransReIDDINOv2ViTB14(NUM_CLASSES, num_cameras=NUM_VERI_CAMERAS, pretrained=True).to(DEVICE)
assert model.vit.patch_embed.num_patches == 256
assert model.vit.embed_dim == 768
print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = int(num_classes)
        self.epsilon = float(epsilon)

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1.0)
            smooth = (1.0 - self.epsilon) * one_hot + self.epsilon / self.num_classes
        return (-smooth * log_probs).sum(dim=1).mean()


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = float(margin)
        self.ranking_loss = nn.MarginRankingLoss(margin=self.margin)

    def forward(self, features, targets):
        features = F.normalize(features.float(), p=2, dim=1)
        distance = torch.cdist(features, features, p=2)
        positive_mask = targets.unsqueeze(0).eq(targets.unsqueeze(1))
        negative_mask = ~positive_mask
        positive_distance = distance.clone()
        positive_distance[~positive_mask] = 0.0
        hardest_positive = positive_distance.max(dim=1).values
        negative_distance = distance.clone()
        negative_distance[~negative_mask] = float("inf")
        hardest_negative = negative_distance.min(dim=1).values
        y = torch.ones_like(hardest_negative)
        return self.ranking_loss(hardest_negative, hardest_positive, y)


id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTHING).to(DEVICE)
triplet_loss_fn = TripletLoss(margin=TRIPLET_MARGIN).to(DEVICE)


def compute_reid_loss(outputs, pids):
    loss_global_ce = id_loss_fn(outputs["global_logits"], pids)
    loss_tri = triplet_loss_fn(outputs["global_feat"].float(), pids)
    loss_jpm_ce = torch.stack([id_loss_fn(logits, pids) for logits in outputs["jpm_logits"]]).mean()
    loss = LAMBDA_GLOBAL_CE * loss_global_ce + LAMBDA_GLOBAL_TRI * loss_tri + LAMBDA_JPM_CE * loss_jpm_ce
    return loss, {
        "global_ce": float(loss_global_ce.detach().cpu()),
        "triplet": float(loss_tri.detach().cpu()),
        "jpm_ce": float(loss_jpm_ce.detach().cpu()),
    }


print("Losses ready: CE label smoothing + TripletLoss(margin=0.3) + 4-group JPM CE")

In [ ]:
LLRD_FACTOR = 0.65


def build_optimizer(current_model):
    vit = current_model.vit
    num_blocks = len(vit.blocks)
    backbone_groups = []

    def _add_group(params, depth_from_top, name):
        params = [p for p in params if p.requires_grad]
        if not params:
            return
        scale = LLRD_FACTOR ** depth_from_top
        lr = BACKBONE_LR * scale
        backbone_groups.append({
            "params": params,
            "lr": lr,
            "base_lr": lr,
            "name": name,
        })

    # Stem: patch_embed + cls_token + pos_embed + (optional) norm_pre - deepest from top
    stem_params = list(vit.patch_embed.parameters())
    if hasattr(vit, "cls_token") and vit.cls_token is not None:
        stem_params.append(vit.cls_token)
    if hasattr(vit, "pos_embed") and vit.pos_embed is not None:
        stem_params.append(vit.pos_embed)
    if hasattr(vit, "norm_pre"):
        stem_params.extend(vit.norm_pre.parameters())
    _add_group(stem_params, depth_from_top=num_blocks + 1, name="stem")

    # Per-block LLRD: block 0 is deepest from top, block N-1 is shallowest from top
    for block_index, block in enumerate(vit.blocks):
        depth_from_top = num_blocks - block_index
        _add_group(list(block.parameters()), depth_from_top=depth_from_top, name=f"block_{block_index}")

    # Final norm + SIE: top of backbone, full backbone_lr
    top_params = list(vit.norm.parameters())
    _add_group(top_params, depth_from_top=0, name="vit_norm")
    _add_group([current_model.sie_embed], depth_from_top=0, name="sie")

    # Heads: BNNeck + classifier + JPM heads - separate head_lr param group
    backbone_param_ids = {id(p) for group in backbone_groups for p in group["params"]}
    head_params = [p for p in current_model.parameters() if p.requires_grad and id(p) not in backbone_param_ids]
    head_group = {"params": head_params, "lr": HEAD_LR, "base_lr": HEAD_LR, "name": "head"}

    optimizer = torch.optim.AdamW(
        backbone_groups + [head_group],
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY,
    )
    print(json.dumps({
        "llrd_factor": LLRD_FACTOR,
        "num_backbone_groups": len(backbone_groups),
        "first_block_lr": backbone_groups[1]["lr"] if len(backbone_groups) > 1 else None,
        "stem_lr": backbone_groups[0]["lr"],
        "vit_norm_lr": next((g["lr"] for g in backbone_groups if g["name"] == "vit_norm"), None),
        "head_lr": HEAD_LR,
    }, indent=2))
    return optimizer


def lr_for_epoch(epoch_index, base_lr):
    if epoch_index < WARMUP_EPOCHS:
        alpha = (epoch_index + 1) / WARMUP_EPOCHS
        return WARMUP_START_LR + alpha * (base_lr - WARMUP_START_LR)
    progress = (epoch_index - WARMUP_EPOCHS + 1) / max(1, EPOCHS - WARMUP_EPOCHS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    return MIN_LR + (base_lr - MIN_LR) * cosine


def set_optimizer_lrs(optimizer, epoch_index):
    for group in optimizer.param_groups:
        group["lr"] = lr_for_epoch(epoch_index, group["base_lr"])


optimizer = build_optimizer(model)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
print(json.dumps({"optimizer": "AdamW", "backbone_lr": BACKBONE_LR, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS}, indent=2))

In [ ]:
optimizer.zero_grad(set_to_none=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(json.dumps({"pre_train_cleanup": "done", "device": DEVICE, "batch_size": BATCH_SIZE}, indent=2))

In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB / {torch.cuda.mem_get_info()[1] / 1024**3:.2f} GB")

In [ ]:
def to_metric_dict(mAP, cmc):
    return {
        "mAP": float(mAP),
        "R1": float(cmc[0]),
        "R5": float(cmc[min(4, len(cmc) - 1)]),
        "R10": float(cmc[min(9, len(cmc) - 1)]),
    }


def eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=50):
    num_query, num_gallery = distmat.shape
    if num_gallery < max_rank:
        max_rank = num_gallery
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc = []
    all_ap = []
    for query_index in range(num_query):
        order = indices[query_index]
        remove = (g_pids[order] == q_pids[query_index]) & (g_camids[order] == q_camids[query_index])
        raw_cmc = matches[query_index][~remove]
        if not np.any(raw_cmc):
            continue
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        num_rel = raw_cmc.sum()
        precision = raw_cmc.cumsum() / (np.arange(len(raw_cmc)) + 1.0)
        all_ap.append(float((precision * raw_cmc).sum() / num_rel))
    return float(np.mean(all_ap)), np.asarray(all_cmc).mean(axis=0)


@torch.no_grad()
def extract_features(loader, concat_patch=False):
    model.eval()
    features = []
    pids = []
    camids = []
    for images, pid, camid, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        cam_tensor = camid.to(DEVICE, non_blocking=True).long()
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            feat = model.inference_features(images, cam_ids=cam_tensor, concat_patch=concat_patch)
            flipped = model.inference_features(torch.flip(images, dims=[3]), cam_ids=cam_tensor, concat_patch=concat_patch)
        feat = F.normalize((feat.float() + flipped.float()) / 2.0, p=2, dim=1)
        features.append(feat.cpu().numpy())
        pids.append(pid.numpy())
        camids.append(camid.numpy())
    return np.concatenate(features, axis=0), np.concatenate(pids), np.concatenate(camids)


def evaluate_base_cls():
    q_features, q_pids, q_camids = extract_features(query_loader, concat_patch=False)
    g_features, g_pids, g_camids = extract_features(gallery_loader, concat_patch=False)
    distmat = 1.0 - np.matmul(q_features, g_features.T)
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    return to_metric_dict(mAP, cmc)


def save_state(path):
    torch.save(model.state_dict(), path)


history = []
best_map = -1.0
best_r1 = -1.0
started_training = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_started = time.time()
    model.train()
    train_sampler.set_epoch(epoch)
    set_optimizer_lrs(optimizer, epoch - 1)
    running_loss = 0.0
    running_parts = defaultdict(float)
    num_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step_index, (images, pids, camids, _) in enumerate(train_loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        pids = pids.to(DEVICE, non_blocking=True).long()
        camids = camids.to(DEVICE, non_blocking=True).long()
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            outputs = model(images, cam_ids=camids)
            loss, loss_parts = compute_reid_loss(outputs, pids)
            scaled_loss = loss / ACCUM_STEPS
        if not torch.isfinite(loss.detach()):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}, step {step_index}: {loss.item()}")
        scaler.scale(scaled_loss).backward()
        if step_index % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += float(loss.detach().cpu())
        for key, value in loss_parts.items():
            running_parts[key] += value
        num_batches += 1

    if num_batches % ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    epoch_minutes = (time.time() - epoch_started) / 60.0
    row = {
        "epoch": epoch,
        "loss": running_loss / max(1, num_batches),
        "loss_parts": {key: value / max(1, num_batches) for key, value in running_parts.items()},
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": next(group["lr"] for group in optimizer.param_groups if group.get("name") == "head"),
        "epoch_minutes": epoch_minutes,
    }

    if epoch in PERIODIC_EVAL_EPOCHS:
        metrics = evaluate_base_cls()
        row["periodic_base_cls"] = metrics
        if metrics["mAP"] > best_map:
            best_map = metrics["mAP"]
            save_state(BEST_MAP_PATH)
        if metrics["R1"] > best_r1:
            best_r1 = metrics["R1"]
            save_state(BEST_R1_PATH)
        print(f"Epoch {epoch:03d}: loss={row['loss']:.4f} base_mAP={metrics['mAP']*100:.2f} R1={metrics['R1']*100:.2f} time={epoch_minutes:.2f}m")
    else:
        print(f"Epoch {epoch:03d}: loss={row['loss']:.4f} time={epoch_minutes:.2f}m")

    history.append(row)
    TRAIN_LOG_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")
    save_state(LAST_PATH)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed_hours = (time.time() - started_training) / 3600.0
print(json.dumps({"training_complete": True, "elapsed_hours": elapsed_hours, "best_base_mAP": best_map, "best_base_R1": best_r1}, indent=2))

In [ ]:
def average_query_expansion(features, k=2, iterations=1):
    current = features.astype(np.float32, copy=True)
    for _ in range(iterations):
        similarity = np.matmul(current, current.T)
        topk = min(k, similarity.shape[1])
        topk_indices = np.argpartition(-similarity, kth=topk - 1, axis=1)[:, :topk]
        expanded = np.zeros_like(current)
        for index in range(current.shape[0]):
            expanded[index] = current[topk_indices[index]].mean(axis=0)
        current = expanded / (np.linalg.norm(expanded, axis=1, keepdims=True) + 1e-12)
    return current


def compute_reranking(q_features, g_features, k1=80, k2=15, lambda_value=0.2):
    all_features = np.concatenate([q_features, g_features], axis=0).astype(np.float32, copy=False)
    all_num = all_features.shape[0]
    query_num = q_features.shape[0]
    original_dist = np.clip(2.0 - 2.0 * np.matmul(all_features, all_features.T), 0.0, None).astype(np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    V = np.zeros((all_num, all_num), dtype=np.float16)
    half_k1 = int(round(k1 / 2.0))

    for index in range(all_num):
        forward = initial_rank[index, :k1 + 1]
        backward = initial_rank[forward, :k1 + 1]
        reciprocal = forward[np.any(backward == index, axis=1)]
        reciprocal_expansion = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, :half_k1 + 1]
            candidate_backward = initial_rank[candidate_forward, :half_k1 + 1]
            candidate_reciprocal = candidate_forward[np.any(candidate_backward == candidate, axis=1)]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.concatenate([reciprocal_expansion, candidate_reciprocal])
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion]).astype(np.float32)
        V[index, reciprocal_expansion] = (weights / (weights.sum() + 1e-12)).astype(np.float16)

    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for index in range(all_num):
            V_qe[index] = V[initial_rank[index, :k2]].mean(axis=0)
        V = V_qe

    inv_index = [np.flatnonzero(V[:, column]) for column in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)
    for index in range(query_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        non_zero = np.flatnonzero(V[index])
        for nz in non_zero:
            related = inv_index[nz]
            temp_min[related] += np.minimum(np.float32(V[index, nz]), V[related, nz].astype(np.float32))
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)
    return jaccard_dist[:, query_num:] * (1.0 - lambda_value) + original_dist[:query_num, query_num:] * lambda_value


def evaluate_feature_row(label, concat_patch=False, aqe_k=None, rerank=False):
    q_features, q_pids, q_camids = extract_features(query_loader, concat_patch=concat_patch)
    g_features, g_pids, g_camids = extract_features(gallery_loader, concat_patch=concat_patch)
    if aqe_k is not None:
        all_features = average_query_expansion(np.concatenate([q_features, g_features], axis=0), k=aqe_k, iterations=1)
        q_features = all_features[: len(q_features)]
        g_features = all_features[len(q_features):]
    if rerank:
        distmat = compute_reranking(q_features, g_features, k1=80, k2=15, lambda_value=0.2)
    else:
        distmat = 1.0 - np.matmul(q_features, g_features.T)
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    metrics = to_metric_dict(mAP, cmc)
    print(f"{label}: mAP={metrics['mAP']*100:.4f}% R1={metrics['R1']*100:.4f}% R5={metrics['R5']*100:.2f}% R10={metrics['R10']*100:.2f}%")
    return {"label": label, "concat_patch": concat_patch, "aqe_k": aqe_k, "rerank": rerank, **metrics}


state_dict = torch.load(BEST_MAP_PATH, map_location="cpu")
load_result = model.load_state_dict(state_dict, strict=True)
print("Loaded best-mAP checkpoint:", load_result)

sample_images, sample_pids, sample_camids, _ = next(iter(query_loader))
sample_images = sample_images[:1].to(DEVICE)
sample_camids = sample_camids[:1].to(DEVICE).long()
with torch.no_grad():
    cls_sample = model.inference_features(sample_images, cam_ids=sample_camids, concat_patch=False)
    concat_sample = model.inference_features(sample_images, cam_ids=sample_camids, concat_patch=True)
assert tuple(cls_sample.shape) == (1, FEATURE_DIM)
assert tuple(concat_sample.shape) == (1, CONCAT_FEATURE_DIM)

eval_rows = [
    evaluate_feature_row("single_flip_cls_base", concat_patch=False, aqe_k=None, rerank=False),
    evaluate_feature_row("single_flip_cls_aqe2_rerank_k1_80_k2_15_lambda_0_2", concat_patch=False, aqe_k=2, rerank=True),
    evaluate_feature_row("concat_patch_flip_aqe2_rerank_k1_80_k2_15_lambda_0_2", concat_patch=True, aqe_k=2, rerank=True),
    evaluate_feature_row("concat_patch_flip_aqe3_rerank_k1_80_k2_15_lambda_0_2", concat_patch=True, aqe_k=3, rerank=True),
]

eval_results = {
    "checkpoint": str(BEST_MAP_PATH),
    "model_name": MODEL_NAME,
    "image_size": IMG_SIZE,
    "rows": eval_rows,
    "winner_band": "WIN if best concat AQE row mAP>=0.9154 and R1>=0.9833; MARGINAL if mAP>=0.905 and R1>=0.980; else FAIL",
}
EVAL_RESULTS_PATH.write_text(json.dumps(eval_results, indent=2), encoding="utf-8")
print(json.dumps(eval_results, indent=2))

In [ ]:
recipe = {
    "spec": "docs/subagent-specs/14r-probe.md",
    "verdict_gate": "best of concat-patch-flip AQE k=2/k=3 rerank rows; WIN >=91.54% mAP and >=98.33% R1",
    "model_name": MODEL_NAME,
    "img_size": IMG_SIZE,
    "patch_size": PATCH_SIZE,
    "num_patches": NUM_PATCHES,
    "feature_dim": FEATURE_DIM,
    "concat_feature_dim": CONCAT_FEATURE_DIM,
    "num_classes": NUM_CLASSES,
    "num_cameras": NUM_VERI_CAMERAS,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "warmup_start_lr": WARMUP_START_LR,
    "min_lr": MIN_LR,
    "sampler": {"p": P_IDS, "k": K_INSTANCES, "batch_size": BATCH_SIZE, "accum_steps": ACCUM_STEPS},
    "losses": {"ce_label_smoothing": LABEL_SMOOTHING, "triplet_margin": TRIPLET_MARGIN, "jpm_groups": 4},
    "normalization": {"mean": DINOV2_MEAN, "std": DINOV2_STD, "resolved_config": resolved_cfg},
    "eval_suite": ["single_flip_cls_base", "single_flip_cls_aqe2_rerank", "concat_patch_flip_aqe2_rerank", "concat_patch_flip_aqe3_rerank"],
    "augmentations": ["resize_bicubic_224", "pad_10", "random_crop_224", "horizontal_flip_0.5", "random_erasing_0.5"],
    "outputs": {
        "best_mAP": str(BEST_MAP_PATH),
        "best_R1": str(BEST_R1_PATH),
        "last": str(LAST_PATH),
        "train_log": str(TRAIN_LOG_PATH),
        "eval_results": str(EVAL_RESULTS_PATH),
        "recipe": str(RECIPE_PATH),
    },
}
RECIPE_PATH.write_text(json.dumps(recipe, indent=2), encoding="utf-8")

summary = {
    "kernel": "gumfreddy/14r-probe-dinov2-veri",
    "eta_hours": 6.0,
    "best_map_checkpoint": str(BEST_MAP_PATH),
    "best_r1_checkpoint": str(BEST_R1_PATH),
    "eval_results_path": str(EVAL_RESULTS_PATH),
    "recipe_path": str(RECIPE_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps({"recipe_saved": str(RECIPE_PATH), "summary_saved": str(SUMMARY_PATH)}, indent=2))